# Tu modelo base en acción

**Sesión 3 · IA Intermedio · EstudIA 2026**

Hoy pones a funcionar un modelo de Hugging Face con tus propias entradas. Elige un camino:

| Camino | Para quién | Qué haces |
|---|---|---|
| **A. Tu proyecto** | Ya tienes un modelo base candidato | Lo cargas y lo pruebas con 3 a 5 ejemplos reales de tu proyecto |
| **B. Práctica** | Todavía no tienes proyecto o modelo | Haces una de las dos tareas de práctica: texto o imagen (o las dos) |

Cuando termines, **entrega el notebook** desde la última celda. La entrega muestra que ejecutaste un
modelo de Hugging Face. No importa si el modelo acierta.

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU: los modelos de este
notebook funcionan sin ella. Si tu modelo es grande y Colab te da una GPU, actívala.

In [ ]:
# Preparación: librerías. Ejecútala una vez.
import os, urllib.request
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s03")

import numpy as np
import torch
from PIL import Image
from transformers import pipeline

DISPOSITIVO = 0 if torch.cuda.is_available() else -1
print("Listo.", f"GPU: {torch.cuda.get_device_name(0)}." if DISPOSITIVO == 0 else "Sin GPU: los modelos de este notebook funcionan igual.")

In [ ]:
# @title Funciones auxiliares: descargar archivos y mostrar resultados
def descargar(nombre):
    """Baja un archivo de la carpeta pública del curso (solo la primera vez) y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", nombre)
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def mostrar(entradas, resultados, cuantas=3):
    """Imprime cada entrada con sus primeras etiquetas y puntajes, sin importar la tarea del modelo."""
    if not isinstance(entradas, (list, tuple)):
        entradas, resultados = [entradas], [resultados]
    for entrada, r in zip(entradas, resultados):
        nombre = entrada if isinstance(entrada, str) else type(entrada).__name__
        print("•", nombre[:90])
        if isinstance(r, dict) and "labels" in r:                      # zero-shot de texto
            r = [{"label": l, "score": s} for l, s in zip(r["labels"], r["scores"])]
        elif isinstance(r, dict):                                      # una sola etiqueta por entrada
            r = [r]
        if isinstance(r, list) and r and isinstance(r[0], dict):
            for p in r[:cuantas]:
                etiqueta = p.get("label") or p.get("token_str") or p.get("entity_group") or p.get("word") or p.get("generated_text") or p.get("summary_text") or p.get("translation_text")
                puntaje = f"{p['score']:.3f}" if "score" in p else ""
                print("   ", etiqueta, puntaje)
        elif hasattr(r, "shape") or (isinstance(r, list) and r and isinstance(r[0], (list, float))):
            print("    un vector de forma", tuple(np.array(r).shape))
        else:
            print("   ", str(r)[:300])

---
## A. Tu proyecto

1. Escribe el nombre de tu modelo en `MODELO`, con la forma `dueño/nombre`. Si lo pusiste en el
   formulario de ideas, la plataforma te lo recuerda en la página de esta actividad.
2. Ejecuta la celda. `pipeline` lee en Hugging Face qué tarea hace el modelo, lo descarga y lo deja
   listo. La primera vez tarda según el tamaño del modelo.

In [ ]:
MODELO = "dueño/nombre-del-modelo"   # cámbialo, por ejemplo: "pysentimiento/robertuito-sentiment-analysis"

if MODELO.startswith("dueño/"):
    modelo = None
    print("Camino A: escribe el nombre de tu modelo en MODELO y vuelve a ejecutar esta celda. "
          "Si vas por el camino B, sigue a la parte B.")
else:
    modelo = pipeline(model=MODELO, device=DISPOSITIVO)
    print("Tarea del modelo:", modelo.task)

3. Escribe en `ENTRADAS` de 3 a 5 ejemplos **reales** de tu proyecto, sin datos personales. Qué
   recibe el modelo depende de su tarea:

| Tarea | Qué va en `ENTRADAS` | Cómo lo llamas |
|---|---|---|
| `text-classification`, `token-classification` | Textos | `modelo(ENTRADAS)` |
| `zero-shot-classification` | Textos | `modelo(ENTRADAS, candidate_labels=["clase 1", "clase 2"])` |
| `fill-mask` | Textos con la palabra que falta marcada con `modelo.tokenizer.mask_token` | `modelo(ENTRADAS)` |
| `image-classification` | Rutas o enlaces de imágenes | `modelo(ENTRADAS)` |
| `zero-shot-image-classification` | Rutas o enlaces de imágenes | `modelo(ENTRADAS, candidate_labels=["clase 1", "clase 2"])` |
| `feature-extraction` | Textos | `modelo(ENTRADAS)`: devuelve vectores, no etiquetas |

Para usar tus propias imágenes, súbelas con el ícono de carpeta, a la izquierda de Colab, y escribe su
nombre, por ejemplo `"mi_foto.jpg"`.

In [ ]:
ENTRADAS = [
    "Escribe aquí un ejemplo real de tu proyecto.",
    "Y otro distinto.",
    "Y uno difícil, de los que crees que le van a costar.",
]
if modelo is not None:
    resultados = modelo(ENTRADAS)
    mostrar(ENTRADAS, resultados)

**Lo que observé** (doble clic aquí y responde en dos o tres líneas):

- ¿Qué hace el modelo con tus ejemplos? ¿Las etiquetas que devuelve sirven para tu proyecto, o tendrías
  que ponerle una cabeza nueva?
- ¿En qué ejemplo se equivocó o dudó? ¿Por qué crees que pasó?
- ¿Qué tendrías que conseguir (datos, etiquetas, otro modelo) para adaptarlo a tu problema?

---
## B. Práctica

Dos tareas **zero-shot**: el modelo no conoce tus clases de antemano. Tú escribes las etiquetas y él
decide cuál encaja mejor con cada entrada, sin entrenar nada.

### B1. Texto: ¿de qué tema es cada mensaje?

Una alcaldía recibe miles de mensajes y quiere enviarlos a la dependencia correcta. Usamos
[`MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli`](https://huggingface.co/MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli),
un modelo pequeño y multilingüe (unos 107 millones de parámetros, licencia MIT).

In [ ]:
MENSAJES = [
    ("El semáforo de la 80 con la 33 lleva tres días apagado y se forman unos trancones terribles.", "movilidad"),
    ("Anoche robaron dos celulares en el paradero del parque y no había ningún policía.", "seguridad"),
    ("En el centro de salud del barrio no hay citas para medicina general hasta el próximo mes.", "salud"),
    ("La escuela de la vereda no tiene profesor de matemáticas desde marzo.", "educación"),
    ("Hace una semana se fue el agua en toda la cuadra y nadie da razón.", "servicios públicos"),
    ("Están quemando basura al lado de la quebrada y el humo llega a todas las casas.", "medio ambiente"),
    ("El metro iba tan lleno esta mañana que no pude subirme en tres trenes.", "movilidad"),
    ("La factura de la luz llegó el doble de lo normal sin ninguna explicación.", "servicios públicos"),
    ("Pusieron cámaras en la ciclorruta, pero siguen robando bicicletas.", "seguridad"),
]
TEMAS = ["movilidad", "seguridad", "salud", "educación", "servicios públicos", "medio ambiente"]

clasificador_texto = pipeline("zero-shot-classification", model="MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli",
                              device=DISPOSITIVO)
resultados = clasificador_texto([m for m, _ in MENSAJES], candidate_labels=TEMAS,
                                hypothesis_template="Este mensaje trata sobre {}.")
aciertos = 0
for (mensaje, tema), r in zip(MENSAJES, resultados):
    aciertos += r["labels"][0] == tema
    print(f"{'✓' if r['labels'][0] == tema else '✗'} {r['labels'][0]} ({r['scores'][0]:.0%}) · esperado: {tema} · {mensaje[:60]}")
print(f"\nAcertó {aciertos} de {len(MENSAJES)}.")

Prueba algo y vuelve a ejecutar:

- Cambia las etiquetas por frases más descriptivas, como `"tránsito y transporte"` o `"agua, luz o gas"`.
- Agrega tus propios mensajes.
- Cambia el modelo por uno más grande,
  [`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`](https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7)
  (unos 279 millones de parámetros). Sin GPU tarda un par de minutos.

**Lo que observé** (doble clic aquí y responde): ¿en qué mensajes se equivocó? ¿Le pondrías tú la
misma etiqueta que esperábamos? ¿Qué cambio mejoró o empeoró el resultado?

### B2. Imagen: ¿qué hay en la foto?

Usamos [`google/siglip2-base-patch16-224`](https://huggingface.co/google/siglip2-base-patch16-224)
(licencia Apache 2.0), un modelo de Google que aprendió con imágenes y sus descripciones en muchos
idiomas. Por eso entiende etiquetas en español.

In [ ]:
FOTOS = ["arepas.jpg", "bandeja-paisa.jpg", "colibri.jpg", "tucan.jpg", "gato.jpg", "metrocable.jpg", "esqueleto.jpg"]
ETIQUETAS = ["comida", "ave", "gato", "perro", "transporte", "edificio"]

clasificador_imagen = pipeline("zero-shot-image-classification", model="google/siglip2-base-patch16-224",
                               device=DISPOSITIVO)
imagenes = [Image.open(descargar(f"img/{f}")).convert("RGB") for f in FOTOS]
resultados = clasificador_imagen(imagenes, candidate_labels=ETIQUETAS, hypothesis_template="Esta es una foto de {}.")

import matplotlib.pyplot as plt
fig, ejes = plt.subplots(1, len(FOTOS), figsize=(18, 3.4))
for eje, imagen, r in zip(ejes, imagenes, resultados):
    eje.imshow(imagen); eje.axis("off")
    eje.set_title("\n".join(f"{p['label']} {p['score']:.3f}" for p in r[:3]), fontsize=9)
plt.show()
print("Fotos de Wikimedia Commons: Instituto de Turismo del Meta (arepas, CC BY-SA 4.0), TitiNicola (bandeja paisa, "
      "CC BY-SA 4.0), Elio Rafael Ariza Ramos (colibrí, CC BY-SA 4.0), Bernard Dupont (tucán, CC BY-SA 2.0), "
      "Alvesgaspar (gato, CC BY-SA 3.0), Bernard Gagnon (Metrocable, CC BY-SA 4.0), Museum of Veterinary Anatomy "
      "FMVZ USP (esqueleto, CC BY-SA 4.0).")

**Mira el orden, no el tamaño del número.** SigLIP no usa softmax: no reparte el 100 % entre las
etiquetas. Le da a cada una una probabilidad propia, como si respondiera sí o no por separado. Por eso
los puntajes son pequeños y no suman 1.

Prueba algo y vuelve a ejecutar:

- Agrega la etiqueta `"esqueleto"`. ¿Qué cambia en la última foto?
- Sube una foto tuya (sin personas) y agrégala a `FOTOS` con su ruta.

**Lo que observé** (doble clic aquí y responde): ¿qué foto le costó más? ¿Qué pasa cuando la respuesta
correcta no está entre las etiquetas?

---
## Entrega

Ejecuta las dos celdas siguientes. La primera solo prepara la función; la segunda lee este notebook
con sus resultados y lo envía a la plataforma. Te pide la clave de la actividad la primera vez.

Si algo falla, descarga el notebook (*Archivo › Descargar › Descargar .ipynb*) y súbelo en la
plataforma, en la actividad “Entrega: tu modelo base en acción”.

In [ ]:
# @title Función para entregar el notebook (no hace falta cambiarla)
import getpass, json, urllib.error, urllib.request

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")
ACTIVIDAD = "“Entrega: tu modelo base en acción”"

def _este_notebook():
    """El notebook tal como se ve en Colab, con sus resultados. Fuera de Colab, el archivo de ESTUDIA_NOTEBOOK."""
    if os.environ.get("ESTUDIA_NOTEBOOK"):
        with open(os.environ["ESTUDIA_NOTEBOOK"], encoding="utf-8") as f:
            return json.load(f)
    try:
        from google.colab import _message
        return _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
    except Exception:
        return None

def entregar():
    """Envía este notebook, con lo que ejecutaste, a la plataforma del curso."""
    global _CLAVE
    cuaderno = _este_notebook()
    if not cuaderno:
        print("No pude leer el notebook desde aquí. Descárgalo (Archivo › Descargar › Descargar .ipynb) y súbelo "
              f"en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    if not _CLAVE:
        _CLAVE = getpass.getpass(f"Pega tu clave (plataforma › {ACTIVIDAD} › Copiar) y oprime Enter: ").strip()
    req = urllib.request.Request(f"{PLATAFORMA}/api/entregas/s03", data=json.dumps({"notebook": cuaderno}).encode(),
                                 method="POST", headers={"Content-Type": "application/json",
                                                         "Authorization": f"Bearer {_CLAVE}",
                                                         "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""
        print("No se pudo entregar:", res.get("error", f"error {e.code}"))
        print(f"Si no se arregla, descarga el notebook y súbelo en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    modelos = ", ".join(res["modelos"]) or "ninguno"
    print(f"Entrega recibida. Celdas de código ejecutadas: {res['ejecutadas']} de {res['celdas']}. "
          f"Modelos de Hugging Face que encontré: {modelos}.")
    for aviso in res.get("avisos", []):
        print("Ojo:", aviso)

In [ ]:
entregar()